# Session 10 · Read a plan, find the skew

Queen City Trip Analytics, a fictional South End, Charlotte analytics firm, bills its fleet customers every month by joining a year of trips to its accounts.

In [ ]:
source ~/dsba6190-live-demo-10/env.sh && echo "$BUCKET" \
  || echo "NOT STAGED. Run prep.ipynb first. Do not run any other cell."

## Step 1 · The billing job, and the first batch submitted

In [ ]:
submit baseline baseline "$OFF"

In [ ]:
gcloud storage du -s "$BASE/data/trips" "$BASE/data/accounts"

In [ ]:
gcloud storage cat "$BASE/jobs/billing.py" | sed -n '/^trips = spark.read/,/agg(F.count/p'

## Step 2 · Read the plan

In [ ]:
waitfor baseline

In [ ]:
planof baseline | head -17

In [ ]:
planof baseline | grep -A2 -E '^\((4|9|14)\) Exchange'

In [ ]:
submit broadcast broadcast "$OFF"

## Step 3 · The stage table, and the straggler

In [ ]:
submit salted salted "$OFF"

In [ ]:
evlog baseline

In [ ]:
biggest baseline

## Step 4 · The cause, in the data

In [ ]:
submit aqe aqe "$AQE_ON"

In [ ]:
bqq "SELECT account_id, COUNT(*) AS trips, ROUND(COUNT(*) / SUM(COUNT(*)) OVER (), 4) AS share
     FROM \`$DS.trips\` GROUP BY 1 ORDER BY 2 DESC LIMIT 5"

## Step 5 · Broadcast: the shuffle is gone

In [ ]:
waitfor broadcast

In [ ]:
planof broadcast | sed -n '2,10p'

In [ ]:
evlog broadcast

## Step 6 · Salted: the straggler is split

In [ ]:
waitfor salted

In [ ]:
planof salted | grep -A2 -E '^\((3|5|10)\) (Project|Exchange|Generate)'

In [ ]:
evlog salted

In [ ]:
biggest salted

## Step 7 · Adaptive query execution: coalesced, not split

In [ ]:
waitfor aqe

In [ ]:
planof aqe | grep -E '^AdaptiveSparkPlan|isFinalPlan'

In [ ]:
finalplan aqe | grep -E 'AQEShuffleRead|SortMergeJoin'

In [ ]:
evlog aqe

In [ ]:
biggest aqe

## Step 8 · The same join in BigQuery

In [ ]:
bqq "SELECT a.tier, DATE_TRUNC(t.pickup_date, MONTH) AS month, COUNT(*) AS trips,
            ROUND(SUM(t.fare), 2) AS revenue
     FROM \`$DS.trips\` t JOIN \`$DS.accounts\` a USING (account_id)
     GROUP BY 1, 2 ORDER BY 1, 2 LIMIT 6"

## Step 9 · Four variants, one answer

In [ ]:
for v in baseline broadcast salted aqe; do
  bq --project_id="$PROJECT" --location=US load --source_format=PARQUET \
     --replace "$PROJECT:${DATASET:?}.out_$v" "$BASE/out/$v/*.parquet" >/dev/null
  bq --project_id="$PROJECT" --location=US --quiet query --use_legacy_sql=false --format=csv \
     "SELECT '$v', COUNT(*), ROUND(SUM(revenue), 2) FROM \`$DS.out_$v\`" | tail -1
done

In [ ]:
for v in baseline broadcast salted aqe; do printf '%-10s ' "$v"; apptime "$v"; done

## Step 10 · Save and number the plan

In [ ]:
planof baseline > skewed-plan.txt
nl -ba -w2 -s '  ' skewed-plan.txt | sed -n '6,16p'

## Step 11 · Teardown

In [ ]:
gcloud storage rm -r "gs://${BUCKET:?}"

In [ ]:
bq --project_id="${PROJECT:?}" rm -r -f -d "${DATASET:?}"

In [ ]:
gcloud dataproc batches list --region "$REGION" --filter="state=RUNNING"
gcloud storage ls | grep -c "${BUCKET:?}"; bq --project_id="$PROJECT" ls | grep -c "${DATASET:?}"